# Football Match Feature Extraction Pipeline
### Gamma Force | M.Sc. Data Science
**Purpose:** Extract and cache all features from match videos.  
Re-run at any time — **already-extracted matches are skipped automatically**.

---
**Extraction outputs (per match half):**
- `event_clips/<match>/<half>_events.pt` — event clips + labels (for Phase 1)
- `tension_clips/<match>/<half>_tension.pt` — clips + tension values (for Phase 3)
- `full_features/<match>/<half>.pt` — full-video CNN features (for general use)
- `extraction_log.csv` — record of every successful extraction

**Cells in order:**
1. Spawn guard
2. GPU check
3. Install dependencies
4. Paths — ✏️ edit here
5. Imports
6. Config — ✏️ edit hyperparameters here
7. Annotation & tracking utilities
8. Video utilities
9. Feature extraction logic
10. Discover matches
11. Run extraction
12. Verify extraction


## Cell 0 — Spawn Guard
Run **first**. Prevents DataLoader deadlock on Python 3.13.

In [25]:
# ── Multiprocessing spawn guard ───────────────────────────────────────────────
# Run this FIRST before any other cell.
import multiprocessing
try:
    multiprocessing.set_start_method("spawn", force=True)
    print("OK  Multiprocessing start method: spawn")
except RuntimeError:
    print("OK  Multiprocessing already set:", multiprocessing.get_start_method())


OK  Multiprocessing start method: spawn


## Cell 1 — Check GPU

In [26]:
import subprocess, torch

print("=" * 55)
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")
print("=" * 55)


  PyTorch version : 2.5.1+cu121
  CUDA available  : True
  GPU             : NVIDIA GeForce RTX 2080 Ti
  VRAM            : 11.5 GB


## Cell 2 — Install Dependencies

In [27]:
%pip install -q av transformers tqdm opencv-python

import torch
TORCH_VER = torch.__version__.split("+")[0]
CUDA_VER  = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print(f"Installing torch-geometric for torch={TORCH_VER}, cuda={CUDA_VER}")
%pip install -q torch-geometric

print("\nOK All dependencies installed.")


Note: you may need to restart the kernel to use updated packages.
Installing torch-geometric for torch=2.5.1, cuda=cu121
Note: you may need to restart the kernel to use updated packages.

OK All dependencies installed.


## Cell 3 — Paths
> ✏️ Edit `BASE` to point at your project folder

In [28]:
import os
from pathlib import Path

# ── ✏️  EDIT THESE PATHS ──────────────────────────────────────────────────────
BASE = Path.home() / "Desktop" / "Gamma Force"   # ← change to your project root
# Colab with Drive:  BASE = Path("/content/drive/MyDrive/DL_Project")

DATASET_DIR = str(BASE / "matches")          # folder containing match sub-folders
CACHE_DIR   = str(BASE / "feature_cache")    # where ALL extracted features are saved

# Sub-directories inside CACHE_DIR (created automatically)
EVENT_CACHE_DIR   = str(BASE / "feature_cache" / "event_clips")
TENSION_CACHE_DIR = str(BASE / "feature_cache" / "tension_clips")
FULL_FEAT_DIR     = str(BASE / "feature_cache" / "full_features")
EXTRACT_LOG       = str(BASE / "feature_cache" / "extraction_log.csv")
# ─────────────────────────────────────────────────────────────────────────────

for d in [EVENT_CACHE_DIR, TENSION_CACHE_DIR, FULL_FEAT_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Dataset dir       : {DATASET_DIR}")
print(f"Event cache       : {EVENT_CACHE_DIR}")
print(f"Tension cache     : {TENSION_CACHE_DIR}")
print(f"Full-feature cache: {FULL_FEAT_DIR}")
print(f"Extraction log    : {EXTRACT_LOG}")

if not Path(DATASET_DIR).exists():
    print(f"\n⚠️  Dataset folder NOT found: {DATASET_DIR}")
    print("   Please check BASE path above.")
else:
    match_dirs = sorted([d for d in Path(DATASET_DIR).iterdir() if d.is_dir()])
    print(f"\n✅ Found {len(match_dirs)} match folders:")
    for d in match_dirs[:5]:
        videos = list(d.glob("*.mkv")) + list(d.glob("*.mp4"))
        print(f"   {d.name}  →  {[v.name for v in videos]}")
    if len(match_dirs) > 5:
        print(f"   ... and {len(match_dirs)-5} more")


Dataset dir       : /home/sysadm/Desktop/Gamma Force/matches
Event cache       : /home/sysadm/Desktop/Gamma Force/feature_cache/event_clips
Tension cache     : /home/sysadm/Desktop/Gamma Force/feature_cache/tension_clips
Full-feature cache: /home/sysadm/Desktop/Gamma Force/feature_cache/full_features
Extraction log    : /home/sysadm/Desktop/Gamma Force/feature_cache/extraction_log.csv

✅ Found 20 match folders:
   2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich  →  ['2_720p.mkv', '1_720p.mkv']
   2015-09-20 - 18-30 Dortmund 3 - 0 Bayer Leverkusen  →  ['2_720p.mkv', '1_720p.mkv']
   2015-10-04 - 18-30 Bayern Munich 5 - 1 Dortmund  →  ['2_720p.mkv', '1_720p.mkv']
   2016-11-26 - 17-30 Eintracht Frankfurt 2 - 1 Dortmund  →  ['2_720p.mkv', '1_720p.mkv']
   2016-12-06 - 22-45 Benfica 1 - 2 Napoli  →  ['2_720p.mkv', '1_720p.mkv']
   ... and 15 more


## Cell 4 — Imports

In [29]:
import os, time, logging, json, csv
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional
from collections import Counter
import random as _r

import torch
import torch.nn as nn
from torch.amp import autocast
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from tqdm.auto import tqdm

try:
    import av
    print("✅ PyAV loaded")
except ImportError:
    av = None
    print("⚠️  PyAV not found — run Cell 2 again")

try:
    import cv2
    HAS_CV2 = True
    print("✅ OpenCV loaded")
except ImportError:
    HAS_CV2 = False
    print("⚠️  OpenCV not found — optical flow disabled")

try:
    import pandas as pd
    print("✅ Pandas loaded")
except ImportError:
    pd = None
    print("⚠️  Pandas not found")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("GammaForce.Extract")
print("\n✅ All imports done.")


✅ PyAV loaded
✅ OpenCV loaded
✅ Pandas loaded

✅ All imports done.


## Cell 5 — Config
> ✏️ Edit hyperparameters here (must match the training notebook)

In [30]:
# ── ✏️  EDIT HYPERPARAMETERS HERE ────────────────────────────────────────────
@dataclass
class ExtractionConfig:
    dataset_dir:       str   = DATASET_DIR
    event_cache_dir:   str   = EVENT_CACHE_DIR
    tension_cache_dir: str   = TENSION_CACHE_DIR
    full_feat_dir:     str   = FULL_FEAT_DIR
    extract_log:       str   = EXTRACT_LOG

    # Video decoding — keep identical to training notebook
    sample_fps:      float = 5.0
    img_size:        int   = 224

    # Event clip window — keep identical to training notebook
    clip_before_sec: float = 4.0
    clip_after_sec:  float = 6.0

    # Optical flow
    use_optical_flow: bool = True

    # CNN backbone (must match training notebook)
    cnn_backbone: str = "resnet18"
    feature_dim:  int = 512

    # Misc
    batch_size:      int  = 64    # frames per backbone forward pass
    mixed_precision: bool = True
    seed:            int  = 42

    # Background clips per half (Other / label=6)
    bg_ratio:           float = 0.5   # bg clips = bg_ratio × real events
    bg_min_gap_sec:     float = 20.0  # min distance from any real event

    # Derived
    clip_len: int = 50
# ─────────────────────────────────────────────────────────────────────────────

cfg = ExtractionConfig()
cfg.clip_len = int((cfg.clip_before_sec + cfg.clip_after_sec) * cfg.sample_fps)

try:
    if not HAS_CV2:
        cfg.use_optical_flow = False
except NameError:
    HAS_CV2 = False
    cfg.use_optical_flow = False

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    cfg.mixed_precision = False

print(f"Device           : {device}")
if device.type == "cuda":
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"Clip duration    : {cfg.clip_before_sec}s + {cfg.clip_after_sec}s = {cfg.clip_len} frames @ {cfg.sample_fps}fps")
print(f"Optical flow     : {cfg.use_optical_flow}")
print(f"Mixed precision  : {cfg.mixed_precision}")


Device           : cuda
GPU              : NVIDIA GeForce RTX 2080 Ti
Clip duration    : 4.0s + 6.0s = 50 frames @ 5.0fps
Optical flow     : True
Mixed precision  : True


## Cell 6 — Annotation & Tracking Utilities

In [31]:
# ── 7-class event mapping ─────────────────────────────────────────────────────
SOCCERNET_TO_IDX = {
    "goal":               2,
    "shots on target":    0,
    "shots off target":   0,
    "corner":             3,
    "foul":               1,
    "offside":            4,
    "direct free-kick":   1,
    "indirect free-kick": 1,
    "yellow card":        1,
    "red card":           1,
    "kick-off":           5,
    "throw-in":           6,
    "ball out of play":   6,
    "clearance":          6,
    "substitution":       6,
}
EVENT_NAMES   = ["Shot", "Foul", "Goal", "Corner", "Offside", "Kickoff", "Other"]
EVENT_TENSION = {2: 1.0, 0: 0.85, 1: 0.6, 3: 0.5, 4: 0.4, 5: 0.3, 6: 0.1}
NUM_CLASSES   = 7

PLAYER_CLASSES        = {1, 2, 3, 4}
TACTICAL_CAMERA_TYPES = {
    "Main camera center", "Main camera left", "Main camera right",
    "Main camera", "Wide", "Medium"
}


def _parse_gametime(gametime_str: str) -> Tuple[int, float]:
    """Parse 'H - MM:SS' -> (half_int, seconds_float)."""
    parts  = gametime_str.split(" - ")
    half   = int(parts[0])
    mm, ss = parts[1].split(":")
    return half, float(int(mm) * 60 + int(ss))


def load_annotations(match_dir) -> dict:
    label_file = Path(match_dir) / "Labels-v2.json"
    if not label_file.exists():
        return {1: [], 2: []}
    with open(label_file) as f:
        data = json.load(f)
    out = {1: [], 2: []}
    for ann in data.get("annotations", []):
        half, video_sec = _parse_gametime(ann["gameTime"])
        label = SOCCERNET_TO_IDX.get(ann["label"].lower(), 6)
        label = min(max(label, 0), NUM_CLASSES - 1)
        out[half].append((video_sec, label))
    return out


def derive_tension_from_events(events: list, query_sec: float,
                                decay_seconds: float = 30.0) -> float:
    tension = 0.0
    for event_sec, label in events:
        dist_sec  = abs(event_sec - query_sec)
        weight    = EVENT_TENSION.get(label, 0.1)
        proximity = max(0.0, 1.0 - (dist_sec / decay_seconds))
        tension   = max(tension, weight * proximity)
    return tension


def compute_optical_flow(frames_tensor: torch.Tensor) -> torch.Tensor:
    T_len = frames_tensor.shape[0]
    H = frames_tensor.shape[2]
    W = frames_tensor.shape[3]
    if not HAS_CV2 or T_len < 2:
        return torch.zeros(max(T_len - 1, 1), 1, H, W)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    frames_np = frames_tensor.permute(0, 2, 3, 1).numpy()
    frames_np = np.clip((frames_np * std + mean) * 255, 0, 255).astype(np.uint8)
    flows = []
    for i in range(T_len - 1):
        g1   = cv2.cvtColor(frames_np[i],   cv2.COLOR_RGB2GRAY)
        g2   = cv2.cvtColor(frames_np[i+1], cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag  = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
        p99  = np.percentile(mag, 99) + 1e-6
        flows.append(torch.tensor(
            np.clip(mag / p99, 0, 1).astype(np.float32)).unsqueeze(0))
    return torch.stack(flows)


print(f"OK Annotation utilities ready.")
print(f"   Classes ({NUM_CLASSES}): {EVENT_NAMES}")


OK Annotation utilities ready.
   Classes (7): ['Shot', 'Foul', 'Goal', 'Corner', 'Offside', 'Kickoff', 'Other']


## Cell 7 — Video Utilities & Backbone

In [32]:
VIDEO_EXTENSIONS = (".mp4", ".mkv", ".avi", ".mov", ".m4v", ".webm")


def find_video_halves(match_dir: Path) -> List[Path]:
    halves = []
    for ext in VIDEO_EXTENSIONS:
        halves.extend(match_dir.glob(f"*{ext}"))
    return sorted(halves)


def discover_matches(dataset_dir: str) -> List[Tuple[Path, Path]]:
    root = Path(dataset_dir)
    if not root.exists():
        log.error(f"Dataset dir not found: {dataset_dir}")
        return []
    matches = []
    for match_dir in sorted(root.iterdir()):
        if not match_dir.is_dir(): continue
        halves = find_video_halves(match_dir)
        if len(halves) < 2:
            log.warning(f"[SKIP] {match_dir.name} — {len(halves)} video(s)")
            continue
        matches.append((halves[0], halves[1]))
    log.info(f"Discovered {len(matches)} matches in {dataset_dir}")
    return matches


def extract_event_clip_raw(video_path: Path, event_sec: float,
                            before_sec: float, after_sec: float,
                            sample_fps: float, img_size: int) -> torch.Tensor:
    """Extract clip centred on event_sec. Returns (T, 3, H, W)."""
    assert av is not None, "PyAV not installed"
    start_sec  = max(0.0, event_sec - before_sec)
    end_sec    = event_sec + after_sec
    target_len = int((before_sec + after_sec) * sample_fps)

    transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    frames    = []
    container = av.open(str(video_path))
    stream    = container.streams.video[0]
    native_fps = float(stream.average_rate)
    step       = max(1, round(native_fps / sample_fps))
    seek_pts   = int(start_sec / stream.time_base)
    container.seek(seek_pts, stream=stream)

    frame_count = 0
    for frame in container.decode(video=0):
        pts_sec = float(frame.pts * stream.time_base)
        if pts_sec < start_sec:
            frame_count += 1; continue
        if pts_sec > end_sec:
            break
        if frame_count % step == 0:
            frames.append(transform(frame.to_image()))
        frame_count += 1
    container.close()

    if not frames:
        return torch.zeros(target_len, 3, img_size, img_size)

    result = torch.stack(frames)
    if result.shape[0] < target_len:
        pad    = torch.zeros(target_len - result.shape[0], 3, img_size, img_size)
        result = torch.cat([result, pad], dim=0)
    return result[:target_len]


def get_video_duration(video_path: Path) -> float:
    """Return video duration in seconds (fallback: 2700.0)."""    
    try:
        container = av.open(str(video_path))
        stream    = container.streams.video[0]
        dur = float(stream.duration * stream.time_base) if stream.duration else 2700.0
        container.close()
        return dur
    except Exception:
        return 2700.0


@torch.no_grad()
def build_backbone(backbone_name: str, feature_dim: int):
    """Build pretrained CNN backbone (output: feature_dim-dim vector per frame)."""    
    if backbone_name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Identity()
    elif backbone_name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Identity()
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")
    return m


@torch.no_grad()
def encode_frames(frames: torch.Tensor, backbone, device, cfg) -> torch.Tensor:
    """Encode (T, 3, H, W) → (T, feature_dim) using backbone in chunks."""    
    backbone.eval()
    feats = []
    for i in range(0, len(frames), cfg.batch_size):
        batch = frames[i:i+cfg.batch_size].to(device)
        with autocast(device_type=device.type, enabled=cfg.mixed_precision):
            feats.append(backbone(batch).cpu().float())
    return torch.cat(feats, dim=0)


print("OK Video utilities and backbone builder ready.")


OK Video utilities and backbone builder ready.


## Cell 8 — Feature Extraction Logic
The core extraction functions.  
**Skip logic:** a match half is skipped if its `.pt` file already exists in the cache directory.


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
#  Extraction Log
#  Tracks which match/half combos have been extracted and when.
# ─────────────────────────────────────────────────────────────────────────────

class ExtractionLog:
    """
    Persistent CSV log of extracted match halves.
    Provides O(1) lookup: is_extracted(match_name, half) -> bool.
    """    
    def __init__(self, log_path: str):
        self.path    = Path(log_path)
        self.records = {}   # (match_name, half) -> dict
        self._load()

    def _load(self):
        if not self.path.exists():
            return
        with open(self.path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                key = (row["match"], int(row["half"]))
                self.records[key] = row

    def is_extracted(self, match_name: str, half: int) -> bool:
        return (match_name, half) in self.records

    def mark_done(self, match_name: str, half: int,
                  n_event_clips: int, n_tension_clips: int,
                  n_full_frames: int, elapsed_sec: float):
        key = (match_name, half)
        self.records[key] = {
            "match":          match_name,
            "half":           half,
            "n_event_clips":  n_event_clips,
            "n_tension_clips":n_tension_clips,
            "n_full_frames":  n_full_frames,
            "elapsed_sec":    f"{elapsed_sec:.1f}",
            "timestamp":      time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        self._save()

    def _save(self):
        if not self.records:
            return
        fields = ["match","half","n_event_clips","n_tension_clips",
                  "n_full_frames","elapsed_sec","timestamp"]
        with open(self.path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            for row in self.records.values():
                writer.writerow(row)

    def summary(self):
        print(f"\n{'='*60}")
        print(f"  Extraction log: {self.path}")
        print(f"  Total extracted halves: {len(self.records)}")
        total_event = sum(int(r['n_event_clips'])   for r in self.records.values())
        total_tens  = sum(int(r['n_tension_clips'])  for r in self.records.values())
        print(f"  Total event clips  : {total_event}")
        print(f"  Total tension clips: {total_tens}")
        print(f"{'='*60}")


# ─────────────────────────────────────────────────────────────────────────────
#  Cache path helpers  (mirror the structure the training notebook expects)
# ─────────────────────────────────────────────────────────────────────────────

def event_cache_path(video_path: Path, cfg) -> Path:
    match_name = video_path.parent.name
    d = Path(cfg.event_cache_dir) / match_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{video_path.stem}_events.pt"


def tension_cache_path(video_path: Path, cfg) -> Path:
    match_name = video_path.parent.name
    d = Path(cfg.tension_cache_dir) / match_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{video_path.stem}_tension.pt"


def full_feature_cache_path(video_path: Path, cfg) -> Path:
    match_name = video_path.parent.name
    d = Path(cfg.full_feat_dir) / match_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{video_path.stem}.pt"


# ─────────────────────────────────────────────────────────────────────────────
#  Half-level extraction
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def extract_half(video_path: Path, half_idx: int,
                 backbone, cfg, device, extraction_log: ExtractionLog):
    """
    Extract all features for one video half.

    Skip condition:
      If extraction_log.is_extracted(match_name, half_idx) is True AND
      all three cache files already exist on disk, skip entirely.

    Returns (n_event_clips, n_tension_clips, n_full_frames) or (0,0,0) if skipped.
    """    
    match_name = video_path.parent.name

    e_path = event_cache_path(video_path, cfg)
    t_path = tension_cache_path(video_path, cfg)
    f_path = full_feature_cache_path(video_path, cfg)

    # ── Skip check ────────────────────────────────────────────────────────────
    already_logged = extraction_log.is_extracted(match_name, half_idx)
    files_exist    = e_path.exists() and t_path.exists() and f_path.exists()

    if already_logged and files_exist:
        log.info(f"[SKIP] {match_name} half {half_idx} — already extracted")
        return 0, 0, 0

    if already_logged and not files_exist:
        log.warning(f"[RE-EXTRACT] {match_name} half {half_idx} — "                    "log says done but files missing. Re-extracting.")

    if not video_path.exists():
        log.error(f"[MISSING] {video_path}")
        return 0, 0, 0

    t_start   = time.time()
    clip_dur  = cfg.clip_before_sec + cfg.clip_after_sec
    target_len = cfg.clip_len

    log.info(f"[START] {match_name}  half={half_idx}  {video_path.name}")

    # ── Load annotations ──────────────────────────────────────────────────────
    anns       = load_annotations(video_path.parent)
    all_events = anns.get(half_idx, [])
    video_len  = get_video_duration(video_path)

    # ──────────────────────────────────────────────────────────────────────────
    # 1. Event clips  (Phase 1 — event detection)
    # ──────────────────────────────────────────────────────────────────────────
    event_samples  = []   # list of {feat, flow, label}
    event_secs_set = set()

    # Real events
    for event_sec, label in all_events:
        if label == 6:   # skip generic Other at source
            continue
        try:
            frames = extract_event_clip_raw(
                video_path, event_sec,
                cfg.clip_before_sec, cfg.clip_after_sec,
                cfg.sample_fps, cfg.img_size)
        except Exception as e:
            log.warning(f"   Event clip failed @{event_sec:.1f}s: {e}")
            continue

        feat = encode_frames(frames, backbone, device, cfg)
        flow = (compute_optical_flow(frames) if cfg.use_optical_flow
                else torch.zeros(max(target_len-1, 1), 1, cfg.img_size, cfg.img_size))

        event_samples.append({"feat": feat, "flow": flow, "label": label})
        event_secs_set.add(event_sec)

    # Background clips (label=6)
    real_event_count = len(event_samples)
    n_bg = max(1, int(real_event_count * cfg.bg_ratio))
    added = attempts = 0
    while added < n_bg and attempts < n_bg * 5:
        attempts += 1
        t_rand = _r.uniform(clip_dur, video_len - clip_dur)
        if any(abs(t_rand - es) < cfg.bg_min_gap_sec for es in event_secs_set):
            continue
        try:
            frames = extract_event_clip_raw(
                video_path, t_rand,
                cfg.clip_before_sec, cfg.clip_after_sec,
                cfg.sample_fps, cfg.img_size)
        except Exception:
            continue
        feat = encode_frames(frames, backbone, device, cfg)
        flow = (compute_optical_flow(frames) if cfg.use_optical_flow
                else torch.zeros(max(target_len-1, 1), 1, cfg.img_size, cfg.img_size))
        event_samples.append({"feat": feat, "flow": flow, "label": 6})
        added += 1

    torch.save(event_samples, e_path)
    log.info(f"   Event clips saved: {len(event_samples)}  → {e_path.name}")

    # ──────────────────────────────────────────────────────────────────────────
    # 2. Tension clips  (Phase 3 — tension regression)
    # ──────────────────────────────────────────────────────────────────────────
    tension_samples = []

    for event_sec, label in all_events:
        if label == 6: continue
        try:
            frames = extract_event_clip_raw(
                video_path, event_sec,
                cfg.clip_before_sec, cfg.clip_after_sec,
                cfg.sample_fps, cfg.img_size)
        except Exception as e:
            log.warning(f"   Tension clip failed @{event_sec:.1f}s: {e}")
            continue
        feat    = encode_frames(frames, backbone, device, cfg)
        tension = derive_tension_from_events(all_events, event_sec)
        tension_samples.append({"feat": feat, "tension": tension})

    # Background tension clips (low tension)
    n_bg_t = max(1, int(len([l for _, l in all_events if l != 6])))
    added = attempts = 0
    while added < n_bg_t and attempts < n_bg_t * 5:
        attempts += 1
        t_rand = _r.uniform(clip_dur, video_len - clip_dur)
        if any(abs(t_rand - es) < cfg.bg_min_gap_sec for es in event_secs_set):
            continue
        try:
            frames = extract_event_clip_raw(
                video_path, t_rand,
                cfg.clip_before_sec, cfg.clip_after_sec,
                cfg.sample_fps, cfg.img_size)
        except Exception:
            continue
        feat    = encode_frames(frames, backbone, device, cfg)
        tension = derive_tension_from_events(all_events, t_rand)
        tension_samples.append({"feat": feat, "tension": tension})
        added += 1

    torch.save(tension_samples, t_path)
    log.info(f"   Tension clips saved: {len(tension_samples)}  → {t_path.name}")

    # ──────────────────────────────────────────────────────────────────────────
    # 3. Full-video CNN features  (Phase 2 / general use)
    # ──────────────────────────────────────────────────────────────────────────
    container  = av.open(str(video_path))
    stream     = container.streams.video[0]
    native_fps = float(stream.average_rate)
    step       = max(1, round(native_fps / cfg.sample_fps))
    transform  = T.Compose([
        T.Resize((cfg.img_size, cfg.img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    raw_frames = []
    for idx, frame in enumerate(container.decode(video=0)):
        if idx % step == 0:
            raw_frames.append(transform(frame.to_image()))
    container.close()

    n_full = 0
    if raw_frames:
        full_feats = encode_frames(torch.stack(raw_frames), backbone, device, cfg)
        torch.save(full_feats, f_path)
        n_full = full_feats.shape[0]
        log.info(f"   Full features saved: {n_full} frames  → {f_path.name}")
    else:
        log.warning(f"   No frames decoded from {video_path.name}")

    elapsed = time.time() - t_start
    extraction_log.mark_done(
        match_name, half_idx,
        len(event_samples), len(tension_samples), n_full, elapsed)

    log.info(f"[DONE] {match_name} half {half_idx}  ({elapsed:.1f}s)")
    return len(event_samples), len(tension_samples), n_full


print("OK Extraction logic defined.")
print("   Skip condition: match already in extraction_log AND .pt files exist.")


OK Extraction logic defined.
   Skip condition: match already in extraction_log AND .pt files exist.


## Cell 9 — Discover Matches

In [34]:
matches = discover_matches(cfg.dataset_dir)

if not matches:
    print("⚠️  No valid matches found. Check DATASET_DIR in Cell 3.")
else:
    print(f"\nFound {len(matches)} matches:")
    for h1, h2 in matches:
        print(f"   {h1.parent.name}  →  [{h1.name}, {h2.name}]")


11:04:29  INFO      Discovered 20 matches in /home/sysadm/Desktop/Gamma Force/matches



Found 20 matches:
   2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich  →  [1_720p.mkv, 2_720p.mkv]
   2015-09-20 - 18-30 Dortmund 3 - 0 Bayer Leverkusen  →  [1_720p.mkv, 2_720p.mkv]
   2015-10-04 - 18-30 Bayern Munich 5 - 1 Dortmund  →  [1_720p.mkv, 2_720p.mkv]
   2016-11-26 - 17-30 Eintracht Frankfurt 2 - 1 Dortmund  →  [1_720p.mkv, 2_720p.mkv]
   2016-12-06 - 22-45 Benfica 1 - 2 Napoli  →  [1_720p.mkv, 2_720p.mkv]
   2016-12-11 - 22-45 Paris SG 2 - 2 Nice  →  [1_720p.mkv, 2_720p.mkv]
   2016-12-17 - 19-00 Guingamp 2 - 1 Paris SG  →  [1_720p.mkv, 2_720p.mkv]
   2017-02-04 - 20-30 Dortmund 1 - 0 RB Leipzig  →  [1_720p.mkv, 2_720p.mkv]
   2017-02-11 - 17-30 Darmstadt 2 - 1 Dortmund  →  [1_720p.mkv, 2_720p.mkv]
   2017-02-14 - 22-45 Paris SG 4 - 0 Barcelona  →  [1_720p.mkv, 2_720p.mkv]
   2017-02-15 - 22-45 Real Madrid 3 - 1 Napoli  →  [1_720p.mkv, 2_720p.mkv]
   2017-03-07 - 22-45 Napoli 1 - 3 Real Madrid  →  [1_720p.mkv, 2_720p.mkv]
   2017-03-12 - 23-00 Lorient 1 - 2 Paris SG  →  [1

## Cell 10 — Run Feature Extraction
This cell is **safe to re-run at any time**.  
Matches already extracted are skipped; only new/missing ones are processed.


In [35]:
if not matches:
    print("⚠️  No matches found. Run Cell 9 first.")
else:
    # ── Initialise extraction log ─────────────────────────────────────────────
    extraction_log = ExtractionLog(cfg.extract_log)

    # ── Count what still needs extraction ────────────────────────────────────
    todo = []
    for half1, half2 in matches:
        for half_idx, vp in [(1, half1), (2, half2)]:
            match_name = vp.parent.name
            e_p = event_cache_path(vp, cfg)
            t_p = tension_cache_path(vp, cfg)
            f_p = full_feature_cache_path(vp, cfg)
            already_done = (extraction_log.is_extracted(match_name, half_idx)
                            and e_p.exists() and t_p.exists() and f_p.exists())
            if not already_done:
                todo.append((half_idx, vp))

    print(f"Total halves    : {len(matches)*2}")
    print(f"Already extracted: {len(matches)*2 - len(todo)}")
    print(f"To extract now  : {len(todo)}")

    if not todo:
        print("\n✅ All matches already extracted — nothing to do.")
    else:
        # ── Build backbone once ───────────────────────────────────────────────
        print(f"\nBuilding {cfg.cnn_backbone} backbone...")
        backbone = build_backbone(cfg.cnn_backbone, cfg.feature_dim).to(device)
        backbone.eval()
        print("OK Backbone ready.")

        total_event_clips   = 0
        total_tension_clips = 0
        total_full_frames   = 0
        skipped             = 0
        errors              = 0

        pbar = tqdm(total=len(todo), desc="Extracting", unit="half")
        for half_idx, vp in todo:
            pbar.set_description(f"{vp.parent.name} half{half_idx}")
            try:
                ne, nt, nf = extract_half(
                    vp, half_idx, backbone, cfg, device, extraction_log)
                if ne == 0 and nt == 0 and nf == 0:
                    skipped += 1
                else:
                    total_event_clips   += ne
                    total_tension_clips += nt
                    total_full_frames   += nf
            except Exception as e:
                log.error(f"FAILED {vp.parent.name} half{half_idx}: {e}")
                errors += 1
            pbar.update(1)
        pbar.close()

        print(f"\n{'='*55}")
        print(f"  Extraction complete")
        print(f"  New event clips   : {total_event_clips}")
        print(f"  New tension clips : {total_tension_clips}")
        print(f"  New full frames   : {total_full_frames}")
        print(f"  Skipped (cached)  : {skipped}")
        print(f"  Errors            : {errors}")
        print(f"{'='*55}")

        extraction_log.summary()


Total halves    : 40
Already extracted: 0
To extract now  : 40

Building resnet18 backbone...
OK Backbone ready.


Extracting:   0%|          | 0/40 [00:00<?, ?half/s]

11:04:31  INFO      [START] 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich  half=1  1_720p.mkv
11:07:02  INFO         Event clips saved: 72  → 1_720p_events.pt
11:10:34  INFO         Tension clips saved: 96  → 1_720p_tension.pt
11:22:29  INFO         Full features saved: 13500 frames  → 1_720p.pt
11:22:29  INFO      [DONE] 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich half 1  (1077.9s)
11:22:29  INFO      [START] 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich  half=2  2_720p.mkv
11:29:45  INFO         Event clips saved: 106  → 2_720p_events.pt
11:37:38  INFO         Tension clips saved: 142  → 2_720p_tension.pt
11:50:00  INFO         Full features saved: 14035 frames  → 2_720p.pt
11:50:00  INFO      [DONE] 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich half 2  (1650.7s)
11:50:00  INFO      [START] 2015-09-20 - 18-30 Dortmund 3 - 0 Bayer Leverkusen  half=1  1_720p.mkv
11:54:51  INFO         Event clips saved: 69  → 1_720p_events.pt
12:00:01  INFO         Tension clips saved


  Extraction complete
  New event clips   : 3068
  New tension clips : 4102
  New full frames   : 549599
  Skipped (cached)  : 0
  Errors            : 0

  Extraction log: /home/sysadm/Desktop/Gamma Force/feature_cache/extraction_log.csv
  Total extracted halves: 197
  Total event clips  : 14039
  Total tension clips: 18784


## Cell 11 — Verify Extraction
Confirm all cache files are present and readable.

In [36]:
print("=" * 55)
print("  Extraction verification")
print("=" * 55)

ok_count    = 0
warn_count  = 0
total_e     = 0
total_t     = 0
label_dist  = Counter()

for half1, half2 in matches:
    for half_idx, vp in [(1, half1), (2, half2)]:
        e_p = event_cache_path(vp, cfg)
        t_p = tension_cache_path(vp, cfg)
        f_p = full_feature_cache_path(vp, cfg)

        missing = [p.name for p in [e_p, t_p, f_p] if not p.exists()]
        if missing:
            print(f"  ⚠️  {vp.parent.name} half{half_idx}: MISSING {missing}")
            warn_count += 1
            continue

        try:
            ev_data  = torch.load(e_p, map_location="cpu", weights_only=False)
            ten_data = torch.load(t_p, map_location="cpu", weights_only=False)
            full_f   = torch.load(f_p, map_location="cpu", weights_only=False)

            ne = len(ev_data)
            nt = len(ten_data)
            nf = full_f.shape[0] if isinstance(full_f, torch.Tensor) else 0

            for s in ev_data:
                label_dist[EVENT_NAMES[s['label']]] += 1

            print(f"  ✅ {vp.parent.name} half{half_idx}: "                  f"events={ne}  tension={nt}  full_frames={nf}")
            total_e += ne
            total_t += nt
            ok_count += 1
        except Exception as e:
            print(f"  ⚠️  {vp.parent.name} half{half_idx}: READ ERROR — {e}")
            warn_count += 1

print(f"\n  Halves OK    : {ok_count}")
print(f"  Halves WARN  : {warn_count}")
print(f"  Total event clips  : {total_e}")
print(f"  Total tension clips: {total_t}")
print(f"\n  Event label distribution:")
for name in EVENT_NAMES:
    n = label_dist.get(name, 0)
    bar = '█' * min(n, 40)
    print(f"    {name:<10}: {n:>5}  {bar}")


  Extraction verification
  ✅ 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich half1: events=72  tension=96  full_frames=13500
  ✅ 2015-08-22 - 16-30 Hoffenheim 1 - 2 Bayern Munich half2: events=106  tension=142  full_frames=14035
  ✅ 2015-09-20 - 18-30 Dortmund 3 - 0 Bayer Leverkusen half1: events=69  tension=92  full_frames=13500
  ✅ 2015-09-20 - 18-30 Dortmund 3 - 0 Bayer Leverkusen half2: events=78  tension=104  full_frames=13500
  ✅ 2015-10-04 - 18-30 Bayern Munich 5 - 1 Dortmund half1: events=69  tension=92  full_frames=13500
  ✅ 2015-10-04 - 18-30 Bayern Munich 5 - 1 Dortmund half2: events=72  tension=96  full_frames=13500
  ✅ 2016-11-26 - 17-30 Eintracht Frankfurt 2 - 1 Dortmund half1: events=84  tension=112  full_frames=14720
  ✅ 2016-11-26 - 17-30 Eintracht Frankfurt 2 - 1 Dortmund half2: events=99  tension=132  full_frames=15180
  ✅ 2016-12-06 - 22-45 Benfica 1 - 2 Napoli half1: events=75  tension=100  full_frames=13500
  ✅ 2016-12-06 - 22-45 Benfica 1 - 2 Napoli half2: eve